In [ ]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.linear_model import LinearRegression
from scipy.optimize import minimize
import scipy.stats as scst
from scipy.stats import mannwhitneyu
from scipy.stats import ttest_ind
from sklearn.mixture import GaussianMixture
import random
import scipy.stats as stats
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Rectangle
from matplotlib import cm
from matplotlib.colors import Normalize
import statsmodels.api as sm
from scipy.optimize import nnls

In [ ]:
pd.set_option('display.max_columns', None)
plt.rcParams['svg.fonttype'] = 'none'

#### Load dataframe with centromere predictions and CN based classification of chromosomes
- merge into a single dataframe

In [ ]:
# load
pred_df = pd.read_csv("pcawg_prediction.tsv", sep="\t")
class_df = pd.read_csv("/g/korbel/hain/centromere/cancer/pcawg_classification.tsv", sep="\t")

# make key
pred_df["KEY"] = pred_df["SAMPLE_TUMOR"] + "_" + pred_df["CHROM"]
class_df["KEY"] = class_df["SAMPLE"] + "_" + class_df["CHROM"]

# merge
pred_df_annot = pd.merge(pred_df, class_df[["CN_CLASS", "LOH_CLASS", "KEY"]], on="KEY", how="inner").copy()

#### Set class for each chromosome for later plotting/stratification...

In [ ]:
col_merged_class = []
col_arm = []
col_project = []

for idx, row in tqdm(pred_df_annot.iterrows(), total=pred_df_annot.shape[0]):

    if row["CN_CLASS"] == "ISO_P":
        col_merged_class.append("ISO")
        col_arm.append("P")
    elif row["CN_CLASS"] == "ISO_Q":
        col_merged_class.append("ISO")
        col_arm.append("Q")
    elif row["CN_CLASS"] == "P_ARM_LOSS":
        col_merged_class.append("ARM_LOSS")
        col_arm.append("P")
    elif row["CN_CLASS"] == "Q_ARM_LOSS":
        col_merged_class.append("ARM_LOSS")
        col_arm.append("Q")
    elif row["CN_CLASS"] == "P_ARM_GAIN":
        col_merged_class.append("ARM_GAIN")
        col_arm.append("P")
    elif row["CN_CLASS"] == "Q_ARM_GAIN":
        col_merged_class.append("ARM_GAIN")
        col_arm.append("Q")
    elif row["CN_CLASS"] == "WHOLE_LOSS":
        col_merged_class.append("WHOLE_LOSS")
        col_arm.append("BOTH")
    elif row["CN_CLASS"] == "WHOLE_GAIN":
        col_merged_class.append("WHOLE_GAIN")
        col_arm.append("BOTH")
    elif row["CN_CLASS"] == "NORMAL":
        col_merged_class.append("NORMAL")
        col_arm.append("BOTH")
    elif row["CN_CLASS"] == "GAIN_4_4":
        col_merged_class.append("GAIN_4")
        col_arm.append("BOTH")
    elif row["CN_CLASS"] == "GAIN_5_5":
        col_merged_class.append("GAIN_5")
        col_arm.append("BOTH")
    elif row["CN_CLASS"] == "GAIN_1_4":
        col_merged_class.append("GAIN_4|1")
        col_arm.append("P")
    elif row["CN_CLASS"] == "GAIN_4_1":
        col_merged_class.append("GAIN_4|1")
        col_arm.append("Q")        
    elif row["CN_CLASS"] == "GAIN_2_4":
        col_merged_class.append("GAIN_4|2")
        col_arm.append("P")
    elif row["CN_CLASS"] == "GAIN_4_2":
        col_merged_class.append("GAIN_4|2")
        col_arm.append("Q")
    elif row["CN_CLASS"] == "GAIN_3_4":
        col_merged_class.append("GAIN_4|3")
        col_arm.append("P")
    elif row["CN_CLASS"] == "GAIN_4_3":
        col_merged_class.append("GAIN_4|3")
        col_arm.append("Q")
    elif row["CN_CLASS"] == "GAIN_5_4":
        col_merged_class.append("GAIN_4|5")
        col_arm.append("P")
    elif row["CN_CLASS"] == "GAIN_4_5":
        col_merged_class.append("GAIN_4|5")
        col_arm.append("Q")
    else:
        col_merged_class.append(row["CN_CLASS"])
        col_arm.append("BOTH")

    col_project.append("PCAWG")

pred_df_annot["MERGED_CN_CLASS"] = col_merged_class
pred_df_annot["ARM"] = col_arm
pred_df_annot["PROJECT"] = col_project

In [ ]:
# plotting orders for full set and set reuduced to major classes
merged_cn_class_order = ["WHOLE_LOSS", "NORMAL", "WHOLE_GAIN", "GAIN_4", "GAIN_5", "ARM_LOSS", "ARM_GAIN", "ISO", "GAIN_4|1", "GAIN_4|2", "GAIN_4|3", "GAIN_4|5", "OTHER"]
merged_cn_class_order_red = ["WHOLE_LOSS", "NORMAL", "WHOLE_GAIN", "GAIN_4", "GAIN_5", "ARM_LOSS", "ARM_GAIN", "ISO"]

### Add TP53 status

In [ ]:
coding_df = pd.read_csv("/g/korbel/hain/centromere/cancer/pcawg_driver_mutations.tsv", sep="\t")
tp53_mut = coding_df[coding_df["gene"].str.contains("TP53")]["sample_id"].unique().tolist()

col_tp53 = []
for _, row in pred_df_annot.iterrows():
    if row["SAMPLE_TUMOR"] in tp53_mut:
        col_tp53.append("MUT")
    else:
        col_tp53.append("WT")
pred_df_annot["TP53"] = col_tp53

<div style="padding: 12px; border-left: 5px solid #f0ad4e; background-color: #fff3cd; color: #664d03;">
  <strong>Warning:</strong> FILTER
</div>

In [ ]:
cutoff_groups = {
    0.12:"GROUP1", 0.1:"GROUP2", 0.2:"GROUP3", 0.3:"GROUP4", 0.4:"GROUP5", 
    0.5:"GROUP6", 0.15:"GROUP7", 0.25:"GROUP8", 0.21:"GROUP9", 0.22:"GROUP10",
    0.23:"GROUP11", 0.24:"GROUP12"
    }

In [ ]:
filter_id = 0.3
pred_df_annot = pred_df_annot[pred_df_annot[cutoff_groups[filter_id]]].copy().reset_index(drop=True)

In [ ]:
pred_df_annot["SAMPLE_TUMOR"].nunique()

In [ ]:
pred_df_annot["HISTOLOGY"].nunique()

In [ ]:
#pred_df_annot = pred_df_annot[pred_df_annot["CLUSTER_H1"] == pred_df_annot["CLUSTER_H2"]]

### Rescale

1.) Fit linear regression to neutral ladder data

In [ ]:
### gather median of each neutral ladder step
x = []
y = []
for y_val, cn_class in zip([0, 1, 2], ["WHOLE_LOSS", "NORMAL", "WHOLE_GAIN"]):
    x.append(pred_df_annot[pred_df_annot["MERGED_CN_CLASS"] == cn_class]["TOTAL_HOR_LENGTH_F_COV_2"].median())
    y.append(y_val)

### fit linear regression
model_ladder = LinearRegression()
model_ladder.fit(np.array(x).reshape(-1, 1), y, sample_weight=[0.2, 1, 0.2]) # set weight of normal CN=2_2 much higher to account for its high abundance

2.) Rescale F value, add 1 to directly create number of centromeres in cell value

In [ ]:
pred_df_annot["SCALED_F"] = 1 + model_ladder.predict(
    pred_df_annot[["TOTAL_HOR_LENGTH_F_COV_2"]].values
)

# Plot landscape - 4B

Make a dataframe that connects which CN states would fit to each type of event

In [ ]:
cn_df_data = {"CLASS":[], "CN":[]}
for m_cn in pred_df_annot["MERGED_CN_CLASS"].unique():
    cn_df_data["CLASS"].append(m_cn)
    if m_cn == "WHOLE_LOSS":
        cn_df_data["CN"].append(["1_1"])
    elif m_cn == "NORMAL":
        cn_df_data["CN"].append(["2_2"])
    elif m_cn == "WHOLE_GAIN":
        cn_df_data["CN"].append(["3_3"])
    elif m_cn == "GAIN_4":
        cn_df_data["CN"].append(["4_4"])
    elif m_cn == "ARM_LOSS":
        cn_df_data["CN"].append(["2_1", "1_2"])
    elif m_cn == "ARM_GAIN":
        cn_df_data["CN"].append(["2_3", "3_2"])
    elif m_cn == "ISO":
        cn_df_data["CN"].append(["1_3", "3_1"])
    else:
        cn_df_data["CN"].append(["2_2"])        

cn_df = pd.DataFrame(cn_df_data)
cn_df.set_index("CLASS", inplace=True, drop=True)

Check for which chromosomes the predicted event and the predicted CN fit 
- if this does not fit it means that the breakpoint is close to but not in the centromere

In [ ]:
col_correct_CN = []
for _, row in pred_df_annot.iterrows():
    if row["CN"] in cn_df.loc[row["MERGED_CN_CLASS"]]["CN"]:
        col_correct_CN.append("YES")
    else:
        col_correct_CN.append("NO")
pred_df_annot["CORRECT_CN"] = col_correct_CN

#### PLOT - Landscape

In [ ]:
def plot_landscape (axis):
    
    ### hide outliers
    quantile_df_data = {"CLASS":[], "LOWER":[], "UPPER":[]}
    for m_cn in pred_df_annot["MERGED_CN_CLASS"].unique():
        quantile_df_data["CLASS"].append(m_cn)
        quantile_df_data["LOWER"].append(pred_df_annot[pred_df_annot["MERGED_CN_CLASS"] == m_cn]["SCALED_F"].quantile(0.02)) # 0.01
        quantile_df_data["UPPER"].append(pred_df_annot[pred_df_annot["MERGED_CN_CLASS"] == m_cn]["SCALED_F"].quantile(0.98)) # 0.99
    quantile_df = pd.DataFrame(quantile_df_data)
    quantile_df.set_index("CLASS", inplace=True, drop=True)

    ### decide if a datapoint is used for the plot -> depends on matching CN and classification and if the datapoint is within the quantile range
    col_include_in_plot = []
    for idx, row in pred_df_annot.iterrows():
        if row["SCALED_F"] >= quantile_df.loc[row["MERGED_CN_CLASS"]]["LOWER"] and row["SCALED_F"] <= quantile_df.loc[row["MERGED_CN_CLASS"]]["UPPER"]:
            col_include_in_plot.append("YES")
        else:
            col_include_in_plot.append("NO")
    pred_df_annot["PLOT"] = col_include_in_plot
    
    ### whole chromosome landscape
    ax_vio_a = sns.violinplot(
        data=pred_df_annot[(pred_df_annot["PLOT"] == "YES") & (pred_df_annot["CORRECT_CN"] =="YES")], 
        x="MERGED_CN_CLASS", 
        y="SCALED_F", 
        ax=axis, 
        order=["WHOLE_LOSS", "NORMAL", "WHOLE_GAIN", "GAIN_4"], 
        inner=None,
        bw_adjust=.9,
        palette=["gainsboro"]#palette=["lightblue", "white", "coral", "red"]
    )

    ### manual boxplots for the whole chromosome CNVs
    box = axis.boxplot(
        [
            pred_df_annot[(pred_df_annot["PLOT"] == "YES") & (pred_df_annot["MERGED_CN_CLASS"] == "WHOLE_LOSS") & (pred_df_annot["CORRECT_CN"] =="YES")]["SCALED_F"],
            pred_df_annot[(pred_df_annot["PLOT"] == "YES") & (pred_df_annot["MERGED_CN_CLASS"] == "NORMAL") & (pred_df_annot["CORRECT_CN"] =="YES")]["SCALED_F"],
            pred_df_annot[(pred_df_annot["PLOT"] == "YES") & (pred_df_annot["MERGED_CN_CLASS"] == "WHOLE_GAIN") & (pred_df_annot["CORRECT_CN"] =="YES")]["SCALED_F"],
            pred_df_annot[(pred_df_annot["PLOT"] == "YES") & (pred_df_annot["MERGED_CN_CLASS"] == "GAIN_4") & (pred_df_annot["CORRECT_CN"] =="YES")]["SCALED_F"]
        ],
        positions=[0, 1, 2, 3],
        patch_artist=True, 
        showfliers=False,
        whiskerprops=dict(visible=False),
        widths=0.08,
        medianprops=dict(color="black", linewidth=1),  # make median line black
        capprops=dict(linewidth=0)   # hides the crossbars (caps)
    )
    # Apply colors
    for patch in box['boxes']:
        patch.set_facecolor("silver")

    ### manual boxplots for the arm level CNVs
    box = axis.boxplot(
        [
            pred_df_annot[(pred_df_annot["PLOT"] == "YES") & (pred_df_annot["CN_CLASS"] == "P_ARM_LOSS") & (pred_df_annot["CORRECT_CN"] =="YES")]["SCALED_F"],
            pred_df_annot[(pred_df_annot["PLOT"] == "YES") & (pred_df_annot["CN_CLASS"] == "Q_ARM_LOSS") & (pred_df_annot["CORRECT_CN"] =="YES")]["SCALED_F"],
            pred_df_annot[(pred_df_annot["PLOT"] == "YES") & (pred_df_annot["CN_CLASS"] == "P_ARM_GAIN") & (pred_df_annot["CORRECT_CN"] =="YES")]["SCALED_F"],
            pred_df_annot[(pred_df_annot["PLOT"] == "YES") & (pred_df_annot["CN_CLASS"] == "Q_ARM_GAIN") & (pred_df_annot["CORRECT_CN"] =="YES")]["SCALED_F"],
            pred_df_annot[(pred_df_annot["PLOT"] == "YES") & (pred_df_annot["CN_CLASS"] == "ISO_P") & (pred_df_annot["CORRECT_CN"] =="YES")]["SCALED_F"],
            pred_df_annot[(pred_df_annot["PLOT"] == "YES") & (pred_df_annot["CN_CLASS"] == "ISO_Q") & (pred_df_annot["CORRECT_CN"] =="YES")]["SCALED_F"]
        ],
        positions=[4 - 0.08 / 2, 4 + 0.08 / 2, 5 - 0.08 / 2, 5 + 0.08 / 2, 6 - 0.08 / 2, 6 + 0.08 / 2],
        widths=0.08,
        patch_artist=True, 
        showfliers=False,
        whiskerprops=dict(visible=False),
        capprops=dict(linewidth=0),
        medianprops=dict(color="black", linewidth=1)
    )
    # Define alternating colors
    colors = [(0, 0, 147/255), (204/255, 116/255, 0)] * 6

    # Apply colors
    for patch, color in zip(box['boxes'], colors):
        patch.set_facecolor(color)

    ax_vio_b = sns.violinplot(
        data=pred_df_annot[(pred_df_annot["MERGED_CN_CLASS"].isin(["ARM_LOSS", "ARM_GAIN", "ISO"])) & (pred_df_annot["PLOT"] == "YES") & (pred_df_annot["CORRECT_CN"] == "YES")], 
        x="MERGED_CN_CLASS", 
        y="SCALED_F", 
        ax=axis, 
        order=["WHOLE_LOSS", "NORMAL", "WHOLE_GAIN", "GAIN_4", "ARM_LOSS", "ARM_GAIN", "ISO"], 
        hue_order=["P", "Q"],
        hue="ARM", 
        split=True, 
        inner=None,
        bw_adjust=.9,
        palette=[(0, 0, 200/255), (1, 150/255, 0)]
    )


    for i in [0, 1, 2, 3, 4, 5]:
        axis.plot([-0.5, -0.5 + len(merged_cn_class_order_red) - 1], [i, i], c="gray", linestyle="dashed", zorder=0)
    axis.set_ylim(-.5, 5.5)
    axis.set_xlim(-.5, 6.5)
    xticks=[0, 1, 2, 3, 4, 5, 6]

    xtick_labels = []
    for mct, mct_label in zip(["WHOLE_LOSS", "NORMAL", "WHOLE_GAIN", "GAIN_4", "ARM_LOSS", "ARM_GAIN", "ISO"], ["Loss", "Normal", "Gain\nCN=3", "Gain\nCN=4", "Arm-loss", "Arm-gain", "Isochrom"]):
        n = pred_df_annot[(pred_df_annot["MERGED_CN_CLASS"] == mct) & (pred_df_annot["CORRECT_CN"] =="YES")].shape[0]
        xtick_labels.append(f"{mct_label}\n(n={n})")
    axis.set_xticks(xticks, xtick_labels)
    axis.set_xlabel("Chromosome CN pattern")
    axis.set_ylabel("Number of centromere in the tumor cell")
    
fig, ax = plt.subplots(figsize=(12.5, 5.5))
plot_landscape (ax)
#plt.savefig(f"../plotting/plots/pcawg_fig4_cancer_figures_landscape.svg")
plt.show()

# Analysis of arm level events
### Fitting to GMM

### Datasets

In [ ]:
### single chromosomes only
datasets = {}

for chrom in pred_df_annot["CHROM"].unique():
    for arm in ["P", "Q"]:
        cn = "1_2"
        if arm == "Q":
            cn = "2_1"
        d = np.array(pred_df_annot[
            (pred_df_annot["MERGED_CN_CLASS"] == "ARM_LOSS") & 
            (pred_df_annot["SCALED_F"].between(0, 4)) & 
            (pred_df_annot["CHROM"] == chrom) &
            (pred_df_annot["CN"] == cn) &
            (pred_df_annot["ARM"].isin([arm]))
        ].drop_duplicates(subset="SAMPLE_TUMOR")["SCALED_F"].values)

        if d.shape[0] >= 5:
            datasets[f"{chrom}_{arm}"] = d
        else:
            print(f"Skipping dataset {chrom}_{arm} with only {d.shape[0]} datapoints")

for chrom in pred_df_annot["CHROM"].unique():
    for arm in ["P", "Q"]:
        cn = "1_2"
        if arm == "Q":
            cn = "2_1"
        
        for tp53_state in ["WT", "MUT"]:
            d = np.array(pred_df_annot[
                (pred_df_annot["MERGED_CN_CLASS"] == "ARM_LOSS") & 
                (pred_df_annot["SCALED_F"].between(0, 4)) & 
                (pred_df_annot["CHROM"] == chrom) &
                (pred_df_annot["CN"] == cn) &
                (pred_df_annot["ARM"].isin([arm])) &
                (pred_df_annot["TP53"] == tp53_state)
            ].drop_duplicates(subset="SAMPLE_TUMOR")["SCALED_F"].values)

            if d.shape[0] >= 5:
                datasets[f"{chrom}_{arm}_{tp53_state}"] = d
            else:
                print(f"Skipping dataset {chrom}_{arm}_{tp53_state} with only {d.shape[0]} datapoints")

### all
datasets["ALL"] = np.array(pred_df_annot[
    (pred_df_annot["MERGED_CN_CLASS"] == "ARM_LOSS") & 
    (pred_df_annot["CN"].isin(["1_2", "2_1"])) &
    pred_df_annot["SCALED_F"].between(0, 4) &
    (pred_df_annot["ARM"].isin(["P", "Q"]))
]["SCALED_F"].values)

datasets["ALL_P"] = np.array(pred_df_annot[
    (pred_df_annot["MERGED_CN_CLASS"] == "ARM_LOSS") & 
    (pred_df_annot["CN"].isin(["1_2"])) &
    pred_df_annot["SCALED_F"].between(0, 4) &
    (pred_df_annot["ARM"].isin(["P"]))
]["SCALED_F"].values)

datasets["ALL_Q"] = np.array(pred_df_annot[
    (pred_df_annot["MERGED_CN_CLASS"] == "ARM_LOSS") & 
    (pred_df_annot["CN"].isin(["2_1"])) &
    pred_df_annot["SCALED_F"].between(0, 4) &
    (pred_df_annot["ARM"].isin(["Q"]))
]["SCALED_F"].values)

In [ ]:
### single chromosomes only
datasets = {}

for chrom in pred_df_annot["CHROM"].unique():
    for arm in ["P", "Q"]:
        cn = "1_2"
        if arm == "Q":
            cn = "2_1"
        d = np.array(pred_df_annot[
            (pred_df_annot["MERGED_CN_CLASS"] == "ARM_LOSS") & 
            (pred_df_annot["SCALED_F"].between(0, 4)) & 
            (pred_df_annot["CHROM"] == chrom) &
            (pred_df_annot["CN"] == cn) &
            (pred_df_annot["ARM"].isin([arm]))
        ].drop_duplicates(subset="SAMPLE_TUMOR")["SCALED_F"].values)

        if d.shape[0] >= 5:
            datasets[f"{chrom}_{arm}"] = d
        else:
            print(f"Skipping dataset {chrom}_{arm} with only {d.shape[0]} datapoints")

In [ ]:
### single chromosomes only
datasets_full = {}

for chrom in pred_df_annot["CHROM"].unique():
    for arm in ["P", "Q"]:
        cn = "1_2"
        if arm == "Q":
            cn = "2_1"
        d = pred_df_annot[
            (pred_df_annot["MERGED_CN_CLASS"] == "ARM_LOSS") & 
            (pred_df_annot["SCALED_F"].between(0, 4)) & 
            (pred_df_annot["CHROM"] == chrom) &
            (pred_df_annot["CN"] == cn) &
            (pred_df_annot["ARM"].isin([arm]))
        ].drop_duplicates(subset="SAMPLE_TUMOR")

        if d.shape[0] >= 5:
            datasets_full[f"{chrom}_{arm}"] = d
        else:
            print(f"Skipping dataset {chrom}_{arm} with only {d.shape[0]} datapoints")

### GMM fit

In [ ]:
"""
Fit a fixed Gaussian Mixture Model (GMM) using non-negative least squares (NNLS).
The function enforces the sum-to-1 constraint on the mixture weights through a heavily weighted penalty row.
"""

def fit_fixed_gmm_nnls (x, loc, scale, penalty=1e3):
    
    ### prepare data
    # sorted datapoint
    data_x = np.sort(x)
    # empirical CDF - each datapoint is 1/len(data_x) up
    data_y = np.arange(1, len(data_x) + 1) / len(data_x)
    
    ### Build design matrix - each column corresponds to the CDF of a Gaussian component at the sorted datapoints
    A = scst.norm.cdf(data_x[:, None], loc=np.asarray(loc), scale=np.asarray(scale))
    
    ### Add sum-to-1 constraint through a heavily weighted row (penalty)
    A_constrained = np.vstack([A, penalty * np.ones(len(loc))])
    
    ### Prepare the constrained target vector for the nnls solver (add the penalty row)
    y_constrained = np.append(data_y, penalty)
    
    ### get weights - solve the optimization problem
    weights, _ = nnls(A_constrained, y_constrained)
    ### normalize weights to sum to 1 - because the nnls solver does not enforce the sum-to-1 constraint exactly only approximatly with the penalty row
    weights = weights / weights.sum()

    return weights

In [ ]:
### set parameters for Gaussian mixture model
n_params = 11
loc=[np.round(i, 1) for i in np.arange(0.6, 2.8, 0.2)]
scale=[0.15] * n_params

columns = [str(x) for x in loc]

In [ ]:
### store nnls fit results (kept separate from df_gmm)
data_nnls = []
index_nnls = []

for dataset in tqdm(datasets):
    w = fit_fixed_gmm_nnls(datasets[dataset], loc, scale, penalty=1e4)
    data_nnls.append(w)
    index_nnls.append(dataset)


df_gmm = pd.DataFrame(data=data_nnls, index=index_nnls, columns=columns)
df_gmm["N"] = [len(datasets[idx]) for idx in df_gmm.index]

In [ ]:
# plotting order
plot_idx = [
    'chr1_P',
    'chr3_P',
    'chr4_P',
    'chr4_Q',
    'chr5_Q',
    'chr6_Q',
    'chr8_P',
    'chr9_P',
    'chr10_P',
    'chr10_Q',
    'chr12_P',
    'chr12_Q',
    'chr16_Q',
    'chr17_P',
    'chr18_P',
    'chr18_Q',
    'chr19_P',
    'chr19_Q',
    'chr20_P']

plot_idx = [x for x in plot_idx if x in df_gmm.index]


In [ ]:
annot_data = []
for i in plot_idx:
    if i.endswith("_Q"):
        arm_color = "gold"
    else:
        arm_color = "cyan"

    n_color = [np.sqrt(df_gmm.loc[i]["N"].max()) / np.sqrt(df_gmm.loc[plot_idx]["N"].max()), 0, 0]
    n_color = [(df_gmm.loc[i]["N"].max()) / (df_gmm.loc[plot_idx]["N"].max()), 0, 0]
    
    annot_data.append({
        "IDX":i,
        "ARM":arm_color,
        #"N":n_color
    })
annot_df = pd.DataFrame(annot_data).set_index("IDX")

In [ ]:
heatmap_plot_idx = []
for idx in df_gmm.index:
    if idx.startswith("chr") and (idx.endswith("_P") or idx.endswith("_Q")) and df_gmm.at[idx, "N"] >= 10: # type: ignore
        heatmap_plot_idx.append(idx)

In [ ]:
heatmap_plot_idx = []
for idx in df_gmm.index:
    if idx.startswith("chr") and (idx.endswith("_P") or idx.endswith("_Q")) and df_gmm.at[idx, "N"] >= 5: # type: ignore
        heatmap_plot_idx.append(idx)

In [ ]:
annot_data = []
for idx in heatmap_plot_idx:
    if idx.endswith("_P"):
        annot_data.append({"IDX":idx, "ARM":(0, 0, 200/255)})
    else:
        annot_data.append({"IDX":idx, "ARM":(1, 150/255, 0)})
annot_df = pd.DataFrame(annot_data).set_index("IDX")

### PLOT - heatmap

In [ ]:
heatmap_plot_idx

In [ ]:
df_gmm["STRONG_TRUNC"] = df_gmm[["0.6", "0.8", "1.0", "1.2", "1.4"]].sum(axis=1)
df_gmm["WEAK_TRUNC"] = df_gmm[["1.6", "1.8"]].sum(axis=1)
df_gmm["NORMAL"] = df_gmm[["2.0", "2.2"]].sum(axis=1)
df_gmm["LONGER"] = df_gmm[["2.4", "2.6"]].sum(axis=1)

g = sns.clustermap(
    df_gmm[["STRONG_TRUNC", "WEAK_TRUNC", "NORMAL", "LONGER"]].loc[heatmap_plot_idx],
    col_cluster=False,
    row_cluster=False,
    cmap="viridis",
    row_colors=annot_df
)
#g.savefig("/g/korbel/hain/centromere/paper_figures/2026_03_26_cancer_figures_heatmap.svg")

In [ ]:
g = sns.clustermap(
    df_gmm[["STRONG_TRUNC", "WEAK_TRUNC", "NORMAL", "LONGER"]].loc[plot_idx],
    col_cluster=False,
    row_cluster=False,
    cmap="viridis",
    row_colors=annot_df
)
#g.savefig("../plotting/plots/pcawg_fig4_heatmap.svg")

### Load CDR position

In [ ]:
cdr_gao_df = pd.read_csv("Gao_CDR_position.tsv", sep="\t")

In [ ]:
data_structuring = []

for idx in df_gmm.index:
    if idx.startswith("chr"):
        chrom = idx.split("_")[0]
        arm = idx.split("_")[1]
        complete = idx
        tp53_state = "NA"
        if complete.endswith("_MUT"):
            tp53_state = "MUT"
        if complete.endswith("_WT"):
            tp53_state = "WT"
        data_structuring.append({
            "CHROM":chrom, 
            "ARM":arm, 
            "COMPLETE":complete, 
            "TP53":tp53_state, 
            "N":df_gmm.at[idx, "N"]})
data_structuring_df = pd.DataFrame(data_structuring)

In [ ]:
### dip position
col_dip_position = []
col_dip_mean = []
col_dip_median = []
col_dip_q25 = []
col_dip_q75 = []

for idx, row in data_structuring_df.iterrows():
    col_dip_position.append(cdr_gao_df[cdr_gao_df["CHROM"] == row["CHROM"]]["CDR_MEDIAN"].values[0])
    col_dip_mean.append(cdr_gao_df[cdr_gao_df["CHROM"] == row["CHROM"]]["CDR_MEAN"].values[0])
    col_dip_median.append(cdr_gao_df[cdr_gao_df["CHROM"] == row["CHROM"]]["CDR_MEDIAN"].values[0])
    col_dip_q25.append(cdr_gao_df[cdr_gao_df["CHROM"] == row["CHROM"]]["CDR_Q25"].values[0])
    col_dip_q75.append(cdr_gao_df[cdr_gao_df["CHROM"] == row["CHROM"]]["CDR_Q75"].values[0])

data_structuring_df["DIP_LOC"] = col_dip_position
data_structuring_df["DIP_MEAN"] = col_dip_mean
data_structuring_df["DIP_MEDIAN"] = col_dip_median
data_structuring_df["DIP_Q25"] = col_dip_q25
data_structuring_df["DIP_Q75"] = col_dip_q75

In [ ]:
col_query = []
for _, row in data_structuring_df.iterrows():
    col_query.append(df_gmm.loc[row["COMPLETE"], "STRONG_TRUNC"])
data_structuring_df["STRONG_TRUNC"] = col_query

### PLOT - Correlation

In [ ]:
def plot_correlation (axis):
    
    cdr_mean = "ASM_MEDIAN"
    
    value = "STRONG_TRUNC"

    n_min = 10
    
    for arm in ["P", "Q"]:
        
        fit_x = []
        fit_y = []
        fit_weights = []
        
        for idx, row in data_structuring_df[data_structuring_df["N"] > n_min].iterrows():
            
            
            
            if not "chr" in row["COMPLETE"]:
                continue
            if not f"_{arm}" in row["COMPLETE"]:
                continue
            if "_MUT" in row["COMPLETE"]:
                continue
            if "_WT" in row["COMPLETE"]:
                continue
            
            chrom = row["CHROM"]
            size = row["N"]
            if arm == "P":
                color = (0, 0, 200/255)
            if arm == "Q":
                color = (1, 150/255, 0)

            
            

            axis.scatter(
                row["STRONG_TRUNC"],
                row["DIP_LOC"],
                color=color,
                #s=size
            )
            ### label chromosome
            
            axis.text(
                row["STRONG_TRUNC"] + 0.01,
                row["DIP_LOC"],
                chrom,
                color=color,
                va="center"
            )
            
            if size > 0:
                fit_x.append(row["STRONG_TRUNC"])
                fit_y.append(row["DIP_LOC"])
                fit_weights.append(size)
            
        fit_x_C = sm.add_constant(fit_x)  # add intercept
        #model = sm.WLS(fit_y, fit_x_C, weights=fit_weights)
        model = sm.WLS(fit_y, fit_x_C)
        results = model.fit()
        
        pred = results.get_prediction(fit_x_C)
        pred_summary = pred.summary_frame(alpha=0.05)   
        
        if len(fit_x_C) < 3:
            continue
        
        print(len(fit_x_C))
             
        sm_df = pd.DataFrame(data={"x":fit_x_C[:,1], "y":pred_summary["mean"], "lower":pred_summary["mean_ci_lower"], "upper":pred_summary["mean_ci_upper"]}).sort_values(by="x")
        
        
        
        #axis.plot(sm_df["x"], sm_df["y"], label=f"{arm.lower()}-arm loss\n$R^2$={np.round(results.rsquared, 3)}, p={np.round(results.pvalues[1], 3)}", color=color)
        axis.plot(sm_df["x"], sm_df["y"], label=f"{arm.lower()} arm loss\n$R^2$={np.round(results.rsquared, 3)}\np={results.pvalues[1]:.2e}", color=color)
        axis.fill_between(sm_df["x"], sm_df["lower"], sm_df["upper"], alpha=0.1, color=color)
    
    axis.set_ylabel("Position of the methylation dip\np-distal     <--->     q-distal")
    axis.set_xlabel("Fraction of arm losses with highly truncated centromere")
    axis.legend(loc='upper center', ncol=2)
    axis.set_ylim(0, 1)
    #axis.set_xlim(0, 0.61)
    

fig, ax = plt.subplots(figsize=(8.5, 5.5))
plot_correlation (ax)
#ax.set_xlim(0, 0.5)
print(ax.get_xlim())
#plt.savefig(f"../plotting/plots/pcawg_fig4_cancer_figures_correlation.svg")
plt.show()

#### Plot Figure S20

In [ ]:
def plot_correlation_tp53 (axis, tp53_state:str):
    
    cdr_mean = "ASM_MEDIAN"
    value = "STRONG_TRUNC"

    n_min = 7
    
    for arm in ["P", "Q"]:
        
        fit_x = []
        fit_y = []
        fit_weights = []
        
        for idx, row in data_structuring_df[data_structuring_df["N"] > n_min].iterrows():


            if not "chr" in row["COMPLETE"]:
                continue
            if not f"_{arm}" in row["COMPLETE"]:
                continue
            if f"_{tp53_state}" not in row["COMPLETE"]:
                continue
            
            chrom = row["CHROM"]
            size = row["N"]
            if arm == "P":
                color = (0, 0, 200/255)
            if arm == "Q":
                color = (1, 150/255, 0)


            axis.scatter(
                row["STRONG_TRUNC"],
                row["DIP_LOC"],
                color=color,
                #s=size
            )
            ### label chromosome
            
            axis.text(
                row["STRONG_TRUNC"] + 0.01,
                row["DIP_LOC"],
                chrom,
                color=color,
                va="center"
            )
            
            if size > 0:
                fit_x.append(row["STRONG_TRUNC"])
                fit_y.append(row["DIP_LOC"])
                fit_weights.append(size)
            
        fit_x_C = sm.add_constant(fit_x)  # add intercept
        #model = sm.WLS(fit_y, fit_x_C, weights=fit_weights)
        model = sm.WLS(fit_y, fit_x_C)
        results = model.fit()
        
        pred = results.get_prediction(fit_x_C)
        pred_summary = pred.summary_frame(alpha=0.05)   
        
        if len(fit_x_C) < 3:
            continue
             
        sm_df = pd.DataFrame(data={"x":fit_x_C[:,1], "y":pred_summary["mean"], "lower":pred_summary["mean_ci_lower"], "upper":pred_summary["mean_ci_upper"]}).sort_values(by="x")
        
        
        
        #axis.plot(sm_df["x"], sm_df["y"], label=f"{arm.lower()}-arm loss\n$R^2$={np.round(results.rsquared, 3)}, p={np.round(results.pvalues[1], 3)}", color=color)
        axis.plot(sm_df["x"], sm_df["y"], label=f"{arm.lower()} arm loss\n$R^2$={np.round(results.rsquared, 3)}\np={np.round(results.pvalues[1], 6)}", color=color)
        axis.fill_between(sm_df["x"], sm_df["lower"], sm_df["upper"], alpha=0.1, color=color)
    
    axis.set_ylabel("Position of the methylation dip\np-distal     <--->     q-distal")
    axis.set_xlabel("Fraction of arm losses with highly truncated centromere")
    axis.legend(loc='upper center', ncol=2)
    axis.set_ylim(0, 1)
    #axis.set_xlim(0, 0.61)


ax_1 = plt.subplot(1, 3, 1)
ax_2 = plt.subplot(1, 3, 2)
ax_3 = plt.subplot(1, 3, 3)

plot_correlation(ax_1)
plot_correlation_tp53(ax_2, tp53_state="WT")
plot_correlation_tp53(ax_3, tp53_state="MUT")

ax_1.set_title("All samples")
ax_2.set_title("TP53 WT")
ax_3.set_title("TP53 MUT")

plt.gcf().set_size_inches(15, 5)
plt.tight_layout()
#plt.savefig("../plotting/plots/pcawg_figure_s20.png")
plt.show()

#### Resample CDR

In [ ]:
### load cdr
cdr_full_df = pd.read_csv("../CDR/rel_cdr_position_full.tsv", sep="\t")

In [ ]:
stable_data = []
for chrom in [f"chr{x}" for x in range(1, 23)]:
    for arm in ["p", "q"]:
        
        median = cdr_full_df[(cdr_full_df["chromosome"] == chrom)]["rel_cdr_position"].median()
        q25 = cdr_full_df[(cdr_full_df["chromosome"] == chrom)]["rel_cdr_position"].quantile(0.25)
        q75 = cdr_full_df[(cdr_full_df["chromosome"] == chrom)]["rel_cdr_position"].quantile(0.75)
        mean = cdr_full_df[(cdr_full_df["chromosome"] == chrom)]["rel_cdr_position"].mean()
        
        classification = "other"
        if arm == "p" and median < 0.35:
            classification = "close"
        elif arm == "p" and median >= 0.65:
            classification = "distant"
        elif arm == "q" and median < 0.35:
            classification = "distant"
        elif arm == "q" and median >= 0.65:
            classification = "close"
        
        stable_data.append({
            "Chromosome": chrom,
            "Median": median,
            "Q25": q25,
            "Q75": q75,
            "Mean": mean,
            "CNA event": f"{arm}-arm loss",
            "CDR-based classification": classification
        })
pd.DataFrame(stable_data)        

In [ ]:
cdr_full_df.groupby("chromosome").describe().loc[[f"chr{x}" for x in range(1, 23)]]

In [ ]:
rng = np.random.default_rng(seed=42)
def sample_cdr (chrom:str, cluster:str):
    return rng.choice(cdr_full_df[cdr_full_df["chromosome"].eq(chrom) & cdr_full_df["cluster"].eq(cluster)]["rel_cdr_position"])

In [ ]:
def correlate_full (x, y):
    x_C = sm.add_constant(x.tolist())
    model = sm.WLS(y.tolist(), x_C)
    results = model.fit()
    
    r2 = results.rsquared
    pvalue = results.pvalues[1]
    slope = results.params[1]
    slope_ci = results.conf_int(alpha=0.05)[1]
    intercept = results.params[0]

    return r2, pvalue, slope, slope_ci, intercept

In [ ]:
n_iter = 10

results_median = []
chrom_sampling_median = []

results_single = []
chrom_sampling_single = []

for i in tqdm(range(n_iter)):
    
    
    n_min = 10
        
    for arm in ["P", "Q"]:
        
        fit_x = []
        fit_y_median = []
        fit_y_single = []
        
        for idx, row in data_structuring_df[data_structuring_df["N"] > n_min].iterrows():
            
            
            if not "chr" in row["COMPLETE"]:
                continue
            if not f"_{arm}" in row["COMPLETE"]:
                continue
            if "_MUT" in row["COMPLETE"]:
                continue
            if "_WT" in row["COMPLETE"]:
                continue
            
            
                        
            chrom = row["CHROM"]
            size = row["N"]
            
            x_value = row["STRONG_TRUNC"]
            y_value = row["DIP_MEDIAN"]
            y_sampling = []
            for idx2, row2 in datasets_full[f"{chrom}_{arm}"].iterrows():
                
                try:
                    dip_h1 = sample_cdr(chrom, row2["CLUSTER_H1"])
                except:
                    dip_h1 = row["DIP_MEDIAN"]
                
                try:
                    dip_h2 = sample_cdr(chrom, row2["CLUSTER_H2"])
                except:
                    dip_h2 = row["DIP_MEDIAN"]

                y_sampling.append(np.random.choice([dip_h1, dip_h2]))

            y_sampling_median = np.median(y_sampling)
            y_sampling_single = rng.choice(y_sampling)
            
            fit_x.append(x_value)
            fit_y_median.append(y_sampling_median)
            fit_y_single.append(y_sampling_single)
            chrom_sampling_median.append({"chrom": chrom, "arm": arm, "x": x_value, "y": y_sampling_median})
            chrom_sampling_single.append({"chrom": chrom, "arm": arm, "x": x_value, "y": y_sampling_single})

        
        for fit_y, res_df in zip([fit_y_median, fit_y_single], [results_median, results_single]):
        
            r2, pvalue, slope, slope_ci, intercept = correlate_full(np.array(fit_x), np.array(fit_y))
        
            res_df.append({
                "iter": i,
                "arm": arm,
                "r2": r2,
                "pvalue": pvalue,
                "slope": slope,
                "slope_ci": slope_ci,
                "intercept": intercept
            })

results_df_median = pd.DataFrame(results_median)
results_df_single = pd.DataFrame(results_single)
chrom_sampling_df_median = pd.DataFrame(chrom_sampling_median)
chrom_sampling_df_single = pd.DataFrame(chrom_sampling_single)

In [ ]:
def plot_sampled_correlation (axis, _sampling_df:pd.DataFrame, _results_df:pd.DataFrame):
    
    ### plot each arm loss separatly
    for arm in ["P", "Q"]:
        
        if arm == "P":
            color = (0, 0, 200/255)
        if arm == "Q":
            color = (1, 150/255, 0)
            
        for chrom in _sampling_df["chrom"].unique():
        
            data_x = _sampling_df[(_sampling_df["chrom"] == chrom) & (_sampling_df["arm"] == arm)]["x"]
            data_y = _sampling_df[(_sampling_df["chrom"] == chrom) & (_sampling_df["arm"] == arm)]["y"]
        
            if pd.isna(data_x).all():
                continue
            
            axis.boxplot(
                data_y,
                positions=[np.mean(data_x)],
                showfliers=False,
                widths=0.02,
                patch_artist=True,
                boxprops={"facecolor": color, "edgecolor": "none", "alpha": 0.5},
                medianprops={"color": "black"},
                whiskerprops={"color": "black"},
                capprops={"color": "none"},
                zorder=1,
            )
            """
            axis.text(
                np.mean(data_x)+0.005,
                np.median(data_y)-1.2*(np.median(data_y)-np.quantile(data_y, 0.25)),
                chrom,
                #ha="center",
                va="top",
                fontsize=8,
                color="black",
                zorder=2
            )
            """
            if (chrom == "chr5" and arm == "Q") or (chrom == "chr4" and arm == "P") or (chrom == "chr12" and arm == "P") or (chrom == "chr17" and arm == "P"):
                axis.text(
                    np.mean(data_x) - 0.01,
                    np.median(data_y),
                    chrom,
                    ha="right",
                    va="center",
                    fontsize=8,
                    color="black",
                    zorder=2
                )
            elif (chrom == "chr12" and arm == "Q"):
                axis.text(
                    np.mean(data_x),
                    np.quantile(data_y, 0),
                    chrom,
                    ha="center",
                    va="center",
                    fontsize=8,
                    color="black",
                    zorder=2
                )
            elif (chrom == "chr10" and arm == "P"):
                axis.text(
                    np.mean(data_x),
                    np.quantile(data_y, 1),
                    chrom,
                    ha="center",
                    va="center",
                    fontsize=8,
                    color="black",
                    zorder=2
                )
            else:
                axis.text(
                    np.mean(data_x) + 0.01,
                    np.median(data_y),
                    chrom,
                    ha="left",
                    va="center",
                    fontsize=8,
                    color="black",
                    zorder=2
                )
            
        # One fitted slope and intercept per resampling round
        slopes = np.asarray(_results_df[_results_df["arm"].eq(arm)]["slope"])
        intercepts = np.asarray(_results_df[_results_df["arm"].eq(arm)]["intercept"])
    
        # Fraction of strongly truncated centromeres (x-axis)
        x_grid = np.linspace(0, 1, 200)
    
        # Predicted regression line from each resampling round
        y_pred = intercepts[:, None] + slopes[:, None] * x_grid[None, :]
    
        # Median fitted line and resampling interval, define resampling cutoff
        cut = 2.5
        y_median = np.median(y_pred, axis=0)
        y_lower = np.percentile(y_pred, cut, axis=0)
        y_upper = np.percentile(y_pred, 100 - cut, axis=0)
        
        # Plot the median regression line and the resampling interval
        #axis.plot(x_grid, y_median, zorder=0, label=f"{arm.lower()}-arm loss: median resampled regression", color=color)
        axis.plot(x_grid, y_median, zorder=0, label=f"{arm.lower()}-arm loss", color=color)
        axis.fill_between(
            x_grid, y_lower, y_upper,
            alpha=0.2,
            zorder=0, color=color,
            #label=f"{arm.lower()}-arm loss: {100 - 2 * cut}% resampling interval"
        )
        axis.legend()
            
    axis.set_ylim(0, 1)
    axis.set_xticks([0, .1, .2, .3, .4, .5, .6, .7, .8, .9, 1], ["0", "0.1", "0.2", "0.3", "0.4", "0.5", "0.6", "0.7", "0.8", "0.9", "1"])
    axis.set_xlim(0, .6)

In [ ]:
fig, axes = plt.subplots(
    nrows=1,
    ncols=4,
    figsize=(12, 4),
    gridspec_kw={"width_ratios": [3, 1, 1, 1]},
    constrained_layout=True,
)

# Panel labels: A–D in the first row and E–H in the second row
for label, ax in zip("ABCD", axes.flat):
    ax.text(
        -0.12, 1.05, label,
        transform=ax.transAxes,
        fontsize=14,
        fontweight="bold",
        va="top",
        ha="right",
    )


plot_sampled_correlation(axes[0], chrom_sampling_df_median, results_df_median)

for i, (metric, label, ylims) in enumerate([("slope", "Slope", (-2.5, 2.5)), ("r2", "R²", (0, 1.1)), ("pvalue", "p-value", (0, 0.15))]):
    sns.boxplot(x="arm", y=metric, data=results_df_median, ax=axes[1 + i], showfliers=False)
    axes[1 + i].set_ylim(ylims)
    axes[1 + i].set_ylabel(label)
    axes[1 + i].set_xlabel("Lost arm")


axes[0].set_ylabel("Position of the methylation dip\np-distal     <--->     q-distal")
axes[0].set_xlabel("Fraction of arm losses with highly truncated centromere")

plt.tight_layout()
#plt.savefig("../plotting/plots/pcawg_cdr_correlation_resampling.png")

In [ ]:
# single
results_df_median.groupby("arm").describe()

In [ ]:
results_df_median[results_df_median["arm"].eq("P") & (results_df_median["pvalue"] < 0.05)].shape[0] / n_iter

In [ ]:
results_df_median[results_df_median["arm"].eq("Q") & (results_df_median["pvalue"] < 0.05)].shape[0] / n_iter

In [ ]:
storage = []
for chrom in chrom_sampling_df_median["chrom"].unique():
    for arm in ["P", "Q"]:
        data = np.array(chrom_sampling_df_median[(chrom_sampling_df_median["chrom"] == chrom) & (chrom_sampling_df_median["arm"] == arm)]["y"])
        if data.size == 0:
            continue
        frac_below_05 = data[data < 0.5].size / data.size
        frac_above_05 = data[data >= 0.5].size / data.size
        frac_below_033 = data[data < 0.33].size / data.size
        frac_above_033 = data[data >= 0.33].size / data.size
        frac_below_066 = data[data < 0.66].size / data.size
        frac_above_066 = data[data >= 0.66].size / data.size
        storage.append({
            "chrom": chrom,
            "arm": arm,
            "frac_below_0.5": frac_below_05,
            "frac_above_0.5": frac_above_05,
            "frac_below_0.33": frac_below_033,
            "frac_above_0.33": frac_above_033,
            "frac_below_0.66": frac_below_066,
            "frac_above_0.66": frac_above_066
        })
data = pd.DataFrame(storage)

col_gao_median = []
for _, row in data.iterrows():
    chrom = row["chrom"]
    col_gao_median.append(cdr_gao_df[cdr_gao_df["CHROM"].eq(chrom)]["CDR_MEDIAN"].values[0])

data["CDR_MEDIAN"] = col_gao_median
data

In [ ]:
my_close = [("chr1", "P"), ("chr9", "P"), ("chr12", "P"), ("chr18", "Q"), ("chr6", "Q")]
my_distant = [("chr3", "P"), ("chr8", "P"), ("chr17", "P"), ("chr18", "P"), ("chr20", "P"), ("chr12", "Q"), ("chr19", "Q"), ("chr16", "Q")]

In [ ]:
col_classification_sampling = []
for _, row in data_structuring_df.iterrows():
    if (row["CHROM"], row["ARM"]) in my_close:
        col_classification_sampling.append("close")
    elif (row["CHROM"], row["ARM"]) in my_distant:
        col_classification_sampling.append("distant")
    else:
        col_classification_sampling.append("other")
data_structuring_df["CLASSIFICATION_SAMPLING"] = col_classification_sampling

In [ ]:
data_structuring_df[data_structuring_df["N"] >= 10].sort_values(by="CLASSIFICATION_SAMPLING")

#### Non-GMM correlation

In [ ]:
n_min = 10
cutoff = 1.5

fig, ax = plt.subplots()

for arm in ["P", "Q"]:
    
    if arm == "P":
        color = (0, 0, 200/255)
    if arm == "Q":
        color = (1, 150/255, 0)
    
    x = []
    y = []

    for _, row in data_structuring_df[(data_structuring_df["N"] > n_min) & (data_structuring_df["ARM"] == arm)].iterrows():
        
        datapoints = np.array(datasets[f"{row['CHROM']}_{row['ARM']}"])
    
        n_datapoints = len(datapoints)
        n_truncated_datapoints = len(datapoints[datapoints < cutoff])
        
        x.append(n_truncated_datapoints / n_datapoints)
        y.append(row["DIP_LOC"])
        ax.text(n_truncated_datapoints / n_datapoints + 0.01, row["DIP_LOC"], s=row["CHROM"])

    
    
    fit_x_C = sm.add_constant(x)  # add intercept
    model = sm.WLS(y, fit_x_C)
    results = model.fit()
            
    pred = results.get_prediction(fit_x_C)
    pred_summary = pred.summary_frame(alpha=0.05)   

    sm_df = pd.DataFrame(data={"x":fit_x_C[:,1], "y":pred_summary["mean"], "lower":pred_summary["mean_ci_lower"], "upper":pred_summary["mean_ci_upper"]}).sort_values(by="x")
            
    ax.plot(sm_df["x"], sm_df["y"], label=f"{arm.lower()} arm loss\n$R^2$={np.round(results.rsquared, 3)}\np={results.pvalues[1]:.3e}", color=color)
    ax.fill_between(sm_df["x"], sm_df["lower"], sm_df["upper"], alpha=0.1, color=color)
    ax.scatter(x, y, color=color)
    

ax.set_ylabel("Position of the methylation dip\np-distal     <--->     q-distal")
ax.set_xlabel("Fraction of arm losses with F <= 1.5")
ax.set_title("Correlation: CDR position vs. truncated centromeres based on F <= 1.5")
ax.set_ylim(0, 1)
ax.legend()
plt.gcf().set_size_inches(7, 4)
plt.tight_layout()
#plt.savefig("../plotting/plots/cancer_figures_correlation.pcawg.F_threshold.png")
plt.show()
plt.close()

#### Leave out of correlation

In [ ]:
def correlate (x, y):
    x_C = sm.add_constant(x.tolist())
    model = sm.WLS(y.tolist(), x_C)
    results = model.fit()
    
    r2 = results.rsquared
    pvalue = results.pvalues[1]
    slope = results.params[1]
    slope_ci = results.conf_int(alpha=0.05)[1]

    return r2, pvalue, slope, slope_ci

In [ ]:
bootstrap_complete_results = []
complete_results = []

for arm in ["P", "Q"]:
        
    corr_data = []
        
    for idx, row in data_structuring_df[data_structuring_df["N"] >= 11].iterrows():

        if not "chr" in row["COMPLETE"]:
            continue
        if not f"_{arm}" in row["COMPLETE"]:
            continue
        if "_MUT" in row["COMPLETE"]:
            continue
        if "_WT" in row["COMPLETE"]:
            continue
        
        chrom = row["CHROM"]
            
        corr_data.append({
            "CHROM": chrom,
            "X": row["STRONG_TRUNC"],
            "Y": row["DIP_LOC"]
        })
    
    bootstrap_results = []
    
    corr_df = pd.DataFrame(corr_data)

    complete_r2, complete_pvalue, complete_slope, complete_slope_ci = correlate(
        corr_df["X"],
        corr_df["Y"]
    )
    for i in range(len(corr_data)):
        fold_index = [x for x in range(len(corr_data)) if not (x == i)]
        r2, pv, slope, slope_ci = correlate(
            corr_df.iloc[fold_index]["X"],
            corr_df.iloc[fold_index]["Y"]
        )
        bootstrap_results.append({
            "CHROM": corr_df.iloc[i]["CHROM"],
            "R2": r2,
            "PValue": pv,
            "Slope": slope,
            "Slope_CI": slope_ci
        })

    bootstrap_df = pd.DataFrame(bootstrap_results)
    bootstrap_df["ARM"] = arm
    bootstrap_complete_results.append(bootstrap_df)
    complete_results.append({
        "ARM": arm,
        "R2": complete_r2,
        "PValue": complete_pvalue,
        "Slope": complete_slope,
        "Slope_CI": complete_slope_ci
    })

bootstrap_complete_df = pd.concat(bootstrap_complete_results, ignore_index=True)
bootstrap_complete_df
complete_results_df = pd.DataFrame(complete_results)
complete_results_df

In [ ]:
for k, (panel_label, value, label) in enumerate([("A", "Slope", "Slope"), ("B", "R2", "R²"), ("C", "PValue", "p-value")]):
    
    ax = plt.subplot(1,3, 1 + k)
    ax.text(-0.1, 1.05, panel_label, transform=ax.transAxes, fontsize=14, fontweight="bold", va="top", ha="right")
    
    x = []
    x_pos = []
    y = []
    color = []
    
    pos = 0
    last_was_p = True
    
    for idx, row in bootstrap_complete_df.iterrows():
        
        if row["ARM"] == "P":
            color.append("tab:blue")
        else:
            color.append("tab:orange")

        if last_was_p and row["ARM"] == "P":
            pos += 1
        elif last_was_p and row["ARM"] == "Q":
            pos += 7
            last_was_p = False
        elif not last_was_p and row["ARM"] == "Q":
            pos += 1
        else:
            pos += 10
            
        x.append(row["CHROM"])
        x_pos.append(pos)
        y.append(row[value])
    
    plot_df = pd.DataFrame({"CHROM": x, "x_pos": x_pos, value: y, "color": color})
    
    ### boxplot
    boxplot = ax.boxplot([
            plot_df[plot_df["color"] == "tab:blue"][value], 
            plot_df[plot_df["color"] == "tab:orange"][value]
        ], 
        positions=[
            plot_df[plot_df["color"] == "tab:blue"]["x_pos"].mean(), 
            plot_df[plot_df["color"] == "tab:orange"]["x_pos"].mean()
        ],
        widths=[
            plot_df[plot_df["color"] == "tab:blue"].shape[0], 
            plot_df[plot_df["color"] == "tab:orange"].shape[0]
        ],
        showfliers=False,
        patch_artist=True,
        boxprops={"edgecolor": "none"},
        medianprops={"color": "black"},
    )
    
    for box, color in zip(boxplot["boxes"], ["tab:blue", "tab:orange"]):
        box.set_facecolor(color)
        box.set_alpha(0.5)
        box.set_edgecolor("none")
    
    ### show data points
    ax.scatter(plot_df[plot_df["color"] == "tab:blue"]["x_pos"], plot_df[plot_df["color"] == "tab:blue"][value], color="tab:blue", label="p-arm loss")
    ax.scatter(plot_df[plot_df["color"] == "tab:orange"]["x_pos"], plot_df[plot_df["color"] == "tab:orange"][value], color="tab:orange", label="q-arm loss")
        
    ### show complete data
    
    complete_p = complete_results_df[complete_results_df["ARM"] == "P"][value].values[0]
    complete_q = complete_results_df[complete_results_df["ARM"] == "Q"][value].values[0]
    ax.scatter(
        plot_df[plot_df["color"] == "tab:blue"]["x_pos"].min() - 2,
        complete_p,
        color="tab:blue",
        label="p-arm complete",
        marker="D"
    )
    
    ax.scatter(
        plot_df[plot_df["color"] == "tab:orange"]["x_pos"].min() - 2,
        complete_q,
        color="tab:orange",
        label="q-arm complete",
        marker="D"
    )
    
    x_pos.append(plot_df[plot_df["color"] == "tab:blue"]["x_pos"].min() - 2)
    x_pos.append(plot_df[plot_df["color"] == "tab:orange"]["x_pos"].min() - 2)
    x.append("complete")
    x.append("complete")
    
    ax.set_xticks(x_pos, x, rotation=90)
    ax.set_ylabel(label)
    ax.set_title(label)
    ax.legend()
    
plt.gcf().set_size_inches(15, 5)
plt.suptitle("Leave-one-chromosome-out sensitivity analysis")
plt.tight_layout()
#plt.savefig("../plotting/plots/leave_one_chromosome_out_sensitivity_analysis.png")
plt.show()

#### Boxplot

In [ ]:
cut = 0.35

In [ ]:
all_chromosomes = [f"chr{i}" for i in range(1, 23)]
chromosomes = [x for x in all_chromosomes if x in data_structuring_df["CHROM"].unique()]

In [ ]:
p = cdr_gao_df[cdr_gao_df["CDR_MEDIAN"] <= cut]["CHROM"].unique().tolist()
q = cdr_gao_df[cdr_gao_df["CDR_MEDIAN"] >= (1 - cut)]["CHROM"].unique().tolist()
mid = [x for x in chromosomes if x not in p and x not in q]

In [ ]:
close = []
distant = []
mid_distance = []

table = []

for _, row in data_structuring_df[data_structuring_df["N"] >= 10].iterrows():
    
    arm = row["ARM"]
    
    arm_long = "p-arm loss"
    if arm == "Q":
        arm_long = "q-arm loss"
    gao_strict = cdr_gao_df[(cdr_gao_df["CHROM"] == row["CHROM"]) & (cdr_gao_df["CNA_EVENT"] == arm_long)]["CDR_BASED_CLASSIFICATION"].values[0]
    
    if arm == "P" and row["CHROM"] in p:
        close.append(row["STRONG_TRUNC"])
        table.append({"CHROM": row["CHROM"], "ARM": row["ARM"], "STRONG_TRUNC": row["STRONG_TRUNC"], "GROUP": "close", "GAO_STRICT": gao_strict})
    elif arm == "P" and row["CHROM"] in q:
        distant.append(row["STRONG_TRUNC"])
        table.append({"CHROM": row["CHROM"], "ARM": row["ARM"], "STRONG_TRUNC": row["STRONG_TRUNC"], "GROUP": "distant", "GAO_STRICT": gao_strict})
    elif arm == "P" and row["CHROM"] in mid:
        mid_distance.append(row["STRONG_TRUNC"])
        table.append({"CHROM": row["CHROM"], "ARM": row["ARM"], "STRONG_TRUNC": row["STRONG_TRUNC"], "GROUP": "mid_distance", "GAO_STRICT": gao_strict})
    elif arm == "Q" and row["CHROM"] in p:
        distant.append(row["STRONG_TRUNC"])
        table.append({"CHROM": row["CHROM"], "ARM": row["ARM"], "STRONG_TRUNC": row["STRONG_TRUNC"], "GROUP": "distant", "GAO_STRICT": gao_strict})
    elif arm == "Q" and row["CHROM"] in q:
        close.append(row["STRONG_TRUNC"])
        table.append({"CHROM": row["CHROM"], "ARM": row["ARM"], "STRONG_TRUNC": row["STRONG_TRUNC"], "GROUP": "close", "GAO_STRICT": gao_strict})
    elif arm == "Q" and row["CHROM"] in mid:
        mid_distance.append(row["STRONG_TRUNC"])
        table.append({"CHROM": row["CHROM"], "ARM": row["ARM"], "STRONG_TRUNC": row["STRONG_TRUNC"], "GROUP": "mid_distance", "GAO_STRICT": gao_strict})   

table_df = pd.DataFrame(table) 

In [ ]:
col_classification_sampling = []
for _, row in table_df.iterrows():
    col_classification_sampling.append(
        data_structuring_df[data_structuring_df["CHROM"].eq(row["CHROM"]) & data_structuring_df["ARM"].eq(row["ARM"])]["CLASSIFICATION_SAMPLING"].values[0]
    )
table_df["CLASSIFICATION_SAMPLING"] = col_classification_sampling

In [ ]:
table_df["CONSISTENT_GROUP"] = table_df.apply(
    lambda row: row["GROUP"] == row["GAO_STRICT"] == row["CLASSIFICATION_SAMPLING"], axis=1
)

In [ ]:
ax1 = plt.subplot(1, 3, 1)
sns.boxplot(
    table_df,
    x="GROUP",
    y="STRONG_TRUNC",
    ax=ax1
)
sns.swarmplot(
    table_df,
    x="GROUP",
    y="STRONG_TRUNC",
    ax=ax1
)

datapoints_close = table_df[table_df["GROUP"] == "close"]["STRONG_TRUNC"].tolist()
datapoints_distant = table_df[table_df["GROUP"] == "distant"]["STRONG_TRUNC"].tolist()
datapoints_mid_distance = table_df[table_df["GROUP"] == "mid_distance"]["STRONG_TRUNC"].tolist()

for x, y, d1, d2 in zip(
    [0, 0, 1], [1, 2, 2], 
    [datapoints_close, datapoints_close, datapoints_mid_distance], [datapoints_mid_distance, datapoints_distant, datapoints_distant]):
    t_stat, p_value = ttest_ind(d1, d2, equal_var=False)
    max_val = np.mean([max(d1), max(d2)])
    ax1.plot([x, y], [1.1 * max_val, 1.1 * max_val])
    ax1.text((x + y) / 2, 1.1 * max_val, f"p = {p_value:.3e}", ha='center')

ax2 = plt.subplot(1, 3, 2)
sns.boxplot(
    table_df,
    x="GAO_STRICT",
    y="STRONG_TRUNC",
    ax=ax2
)
sns.swarmplot(
    table_df,
    x="GAO_STRICT",
    y="STRONG_TRUNC",
    ax=ax2
)

t_stat, p_value = ttest_ind(
    table_df[table_df["GAO_STRICT"] == "close"]["STRONG_TRUNC"], table_df[table_df["GAO_STRICT"] == "distant"]["STRONG_TRUNC"], equal_var=False)
print(p_value)
ax2.plot([0, 1], [0.6, 0.6])
ax2.text(.5, 0.61, f"p = {p_value:.3e}", ha='center')





ax3 = plt.subplot(1, 3, 3)
sns.boxplot(
    table_df,
    x="CLASSIFICATION_SAMPLING",
    y="STRONG_TRUNC",
    ax=ax3
)
sns.swarmplot(
    table_df,
    x="CLASSIFICATION_SAMPLING",
    y="STRONG_TRUNC",
    ax=ax3
)
t_stat, p_value = ttest_ind(
    table_df[table_df["CLASSIFICATION_SAMPLING"] == "close"]["STRONG_TRUNC"], table_df[table_df["CLASSIFICATION_SAMPLING"] == "distant"]["STRONG_TRUNC"], equal_var=False)
print(p_value)

plt.gcf().set_size_inches(15, 3)
plt.tight_layout()
#plt.savefig(f"boxplot.pub_{filter_id}.png")
plt.show()

### PLOT - Boxplot

In [ ]:
from scipy.stats import mannwhitneyu

In [ ]:
table_df.groupby("GROUP").describe()

In [ ]:
0.400472 / 0.165555	

In [ ]:
for y in [5]:
    fig, ax = plt.subplots(figsize=(3.5, y))

    sns.boxplot(
        data=table_df,
        x="GROUP",
        y="STRONG_TRUNC",
        order=["close", "distant"],
        color="lightgray",
        linewidth=0,  # removes only the box outline
        whiskerprops={"color": "black", "linewidth": 1.5},
        capprops={"color": "black", "linewidth": 1.5},
        medianprops={"color": "black", "linewidth": 1.5},
        ax=ax,
    )

    sns.swarmplot(
        data=table_df,
        x="GROUP",
        y="STRONG_TRUNC",
        order=["close", "distant"],
        color="black",
        size=7,
        ax=ax,
    )

    ax.set_xticks([0, 1], ["Close", "Distant"])

    stat, p_value = mannwhitneyu(
        table_df.loc[table_df["GROUP"] == "close", "STRONG_TRUNC"],
        table_df.loc[table_df["GROUP"] == "distant", "STRONG_TRUNC"],
        alternative="two-sided",
        method="auto"
    )

    ax.plot([0, 0, 1, 1], [0.58, 0.6, 0.6, 0.58], color="black")
    ax.text(0.5, 0.61, f"p = {p_value:.2e}", ha="center")
    ax.set_ylim(-0.05, 0.65)
    ax.set_xlabel("CDR position relative to arm loss")
    ax.set_ylabel("Contribution of\nstrongly truncated centromeres to arm losses")

    plt.tight_layout()
    #plt.savefig("../plotting/plots/pcawg_arm_level_analysis.png")
    #plt.savefig(f"../plotting/plots/pcawg_arm_level_analysis.{y}.svg")
    plt.show()

#### Supplement variant

In [ ]:
table_df

In [ ]:
ax1 = plt.subplot(1, 2, 1)
ax2 = plt.subplot(1, 2, 2)

sns.boxplot(
    data=table_df,
    x="GROUP",
    y="STRONG_TRUNC",
    order=["close", "mid_distance", "distant"],
    color="lightgray",
    linewidth=0,  # removes only the box outline
    whiskerprops={"color": "black", "linewidth": 1.5},
    capprops={"color": "black", "linewidth": 1.5},
    medianprops={"color": "black", "linewidth": 1.5},
    ax=ax1,
)

sns.swarmplot(
    data=table_df,
    x="GROUP",
    y="STRONG_TRUNC",
    order=["close", "mid_distance", "distant"],
    color="black",
    size=7,
    ax=ax1,
)

ax1.set_xticks([0, 1, 2], ["Close", "Intermediate", "Distant"])

stat, p_value = mannwhitneyu(
    table_df.loc[table_df["GROUP"] == "close", "STRONG_TRUNC"],
    table_df.loc[table_df["GROUP"] == "distant", "STRONG_TRUNC"],
    alternative="two-sided",
    method="auto"
)

ax1.plot([0, 0, 2, 2], [0.68, 0.7, 0.7, 0.68], color="black")
ax1.text(1, 0.71, f"p = {p_value:.2e}", ha="center")
ax1.set_ylim(-0.05, 0.75)
ax1.set_xlabel("CDR position relative to arm loss")
ax1.set_ylabel("Contribution of\nstrongly truncated centromeres to arm losses")

ax1.plot([0, 0, 1, 1], [0.48, 0.5, 0.5, 0.48], color="black")
ax1.text(0.5, 0.51, f"n.s.", ha="center")

ax1.plot([1, 1, 2, 2], [0.58, 0.6, 0.6, 0.58], color="black")
ax1.text(1.5, 0.61, f"n.s.", ha="center")


######################################


sns.boxplot(
    data=table_df,
    x="GAO_STRICT",
    y="STRONG_TRUNC",
    order=["close", "distant"],
    color="lightgray",
    linewidth=0,  # removes only the box outline
    whiskerprops={"color": "black", "linewidth": 1.5},
    capprops={"color": "black", "linewidth": 1.5},
    medianprops={"color": "black", "linewidth": 1.5},
    ax=ax2,
)

sns.swarmplot(
    data=table_df,
    x="GAO_STRICT",
    y="STRONG_TRUNC",
    order=["close", "distant"],
    color="black",
    size=7,
    ax=ax2,
)

ax2.set_xticks([0, 1], ["Close", "Distant"])

stat, p_value = mannwhitneyu(
    table_df.loc[table_df["GAO_STRICT"] == "close", "STRONG_TRUNC"],
    table_df.loc[table_df["GAO_STRICT"] == "distant", "STRONG_TRUNC"],
    alternative="two-sided",
    method="auto"
)

ax2.plot([0, 0, 1, 1], [0.58, 0.6, 0.6, 0.58], color="black")
ax2.text(0.5, 0.61, f"p = {p_value:.2e}", ha="center")
ax2.set_ylim(-0.05, 0.75)
ax2.set_xlabel("CDR position relative to arm loss")
ax2.set_ylabel("Contribution of\nstrongly truncated centromeres to arm losses")


ax1.set_title("Three-category\nCDR-position stratification")
ax2.set_title("Two-category\nCDR-position stratification")


plt.tight_layout()
#plt.savefig("../plotting/plots/pcawg_arm_level_analysis_supplement.png")
plt.show()

#### Figure S19

In [ ]:
datasets.keys()

In [ ]:
f_array = []
label_array = []
for key in datasets.keys():
    f_array += list(datasets[key])
    label_array += [key] * len(datasets[key])




fig_s19_plot = pd.DataFrame(data={"GROUP": label_array, "VALUE": f_array}).set_index("GROUP")

### make close and distant group
close_group = list(table_df[table_df["GROUP"] == "close"]["CHROM"] + "_" + table_df[table_df["GROUP"] == "close"]["ARM"])
distant_group = list(table_df[table_df["GROUP"] == "distant"]["CHROM"] + "_" + table_df[table_df["GROUP"] == "distant"]["ARM"])

close_df = fig_s19_plot.loc[close_group]
distant_df = fig_s19_plot.loc[distant_group]

close_df["GROUP"] = "close"
distant_df["GROUP"] = "distant"

fig_s19_plot = pd.concat([fig_s19_plot, close_df.set_index("GROUP"), distant_df.set_index("GROUP")])

In [ ]:
ax_top = plt.subplot(2, 1, 1)
ax_bottom = plt.subplot(2, 1, 2)

sns.violinplot(
    fig_s19_plot.loc[heatmap_plot_idx],
    x="GROUP",
    y="VALUE",
    ax=ax_top
)
sns.violinplot(
    fig_s19_plot.loc[["ALL", "ALL_P", "ALL_Q", "close", "distant"]],
    x="GROUP",
    y="VALUE",
    ax=ax_bottom
)


pval = stats.kstest(
    fig_s19_plot.loc[close_group, "VALUE"],
    fig_s19_plot.loc[distant_group, "VALUE"],
).pvalue

ax_bottom.plot([3, 3, 4, 4], [4, 4.5, 4.5, 4], color="black")
ax_bottom.text(3.5, 4.6, f"p = {pval:.2e}", ha="center")


for ax, num in zip([ax_top, ax_bottom], [23, 5]):
    ax.set_xlabel("Group")
    ax.set_ylabel("# of centromeres in tumor cell")
    ax.set_ylim(-0.2, 4.9)
    ax.set_xlim(-0.5, num-0.5)
    for cn in [0, 1, 2, 3, 4]:
        ax.plot([-0.5, num-0.5], [cn, cn], color="lightgray", zorder=0)
    
plt.gcf().set_size_inches(20, 7)
plt.tight_layout()
#plt.savefig("../plotting/plots/pcawg_s19.png")
plt.show()

## Table S_arm_loss_numbers

In [ ]:
df_gmm[df_gmm["N"] >= 10][["N", "STRONG_TRUNC", "WEAK_TRUNC", "NORMAL", "LONGER"]]

# Section about chr4, chr5 and chr10 special analysis

The idea is that the chromosomes 4, 5 and 10 have a distinct bimodal distribution of CDR depending on the architecture. Therefore, they can be split into groups where the CDR is close or distant to the lost arm in cases where a samples e.g. shows homozygously an architecture that is close to the lost arm.

1.) I select all arm loss event for the three chromosomes

In [ ]:
chr4p = pred_df_annot[
    (pred_df_annot["MERGED_CN_CLASS"] == "ARM_LOSS") & 
    (pred_df_annot["SCALED_F"].between(0, 4)) & 
    (pred_df_annot["CHROM"] == "chr4") &
    (pred_df_annot["CN"] == "1_2") &
    (pred_df_annot["ARM"].isin(["P"]))
].drop_duplicates(subset="SAMPLE_TUMOR")

chr4q = pred_df_annot[
    (pred_df_annot["MERGED_CN_CLASS"] == "ARM_LOSS") & 
    (pred_df_annot["SCALED_F"].between(0, 4)) & 
    (pred_df_annot["CHROM"] == "chr4") &
    (pred_df_annot["CN"] == "2_1") &
    (pred_df_annot["ARM"].isin(["Q"]))
].drop_duplicates(subset="SAMPLE_TUMOR")

chr5p = pred_df_annot[
    (pred_df_annot["MERGED_CN_CLASS"] == "ARM_LOSS") & 
    (pred_df_annot["SCALED_F"].between(0, 4)) & 
    (pred_df_annot["CHROM"] == "chr5") &
    (pred_df_annot["CN"] == "1_2") &
    (pred_df_annot["ARM"].isin(["P"]))
].drop_duplicates(subset="SAMPLE_TUMOR")

chr5q = pred_df_annot[
    (pred_df_annot["MERGED_CN_CLASS"] == "ARM_LOSS") & 
    (pred_df_annot["SCALED_F"].between(0, 4)) & 
    (pred_df_annot["CHROM"] == "chr5") &
    (pred_df_annot["CN"] == "2_1") &
    (pred_df_annot["ARM"].isin(["Q"]))
].drop_duplicates(subset="SAMPLE_TUMOR")

chr10p = pred_df_annot[
    (pred_df_annot["MERGED_CN_CLASS"] == "ARM_LOSS") & 
    (pred_df_annot["SCALED_F"].between(0, 4)) & 
    (pred_df_annot["CHROM"] == "chr10") &
    (pred_df_annot["CN"] == "1_2") &
    (pred_df_annot["ARM"].isin(["P"]))
].drop_duplicates(subset="SAMPLE_TUMOR")

chr10q = pred_df_annot[
    (pred_df_annot["MERGED_CN_CLASS"] == "ARM_LOSS") & 
    (pred_df_annot["SCALED_F"].between(0, 4)) & 
    (pred_df_annot["CHROM"] == "chr10") &
    (pred_df_annot["CN"] == "2_1") &
    (pred_df_annot["ARM"].isin(["Q"]))
].drop_duplicates(subset="SAMPLE_TUMOR")

**2.) Split into close and distant and undefined arm losses**

I use clearly p- or q-arm adjacent architectures only:

| Chromosome | p-adjacent | q-adjacent |
|---|---|---|
| chr4 | 1081 | 1075, 1079, 1072 |
| chr5 | 1106 | 1103 |
| chr10 | 1114, 1102 | 968, 1088, 1108, 1110 |

Based on those and the lost arm I classify events, the remaining architectures (e.g. where at least one architecture is not clearly arm-adjacent or has different adjacencies) are merged into the other category

In [ ]:
### close
close = pd.concat([
    chr4p[
        chr4p["CLUSTER_H1"].isin(["chr4_1081"]) & 
        chr4p["CLUSTER_H2"].isin(["chr4_1081"])
    ],
    chr4q[
        chr4q["CLUSTER_H1"].isin(["chr4_1072", "chr4_1075", "chr4_1079"]) & 
        chr4q["CLUSTER_H2"].isin(["chr4_1072", "chr4_1075", "chr4_1079"])
    ],
    chr5p[
        chr5p["CLUSTER_H1"].isin(["chr5_1106"]) & 
        chr5p["CLUSTER_H2"].isin(["chr5_1106"])
    ],
    chr5q[
        chr5q["CLUSTER_H1"].isin(["chr5_1103"]) & 
        chr5q["CLUSTER_H2"].isin(["chr5_1103"])
    ],
    chr10p[
        chr10p["CLUSTER_H1"].isin(["chr10_1102"]) & 
        chr10p["CLUSTER_H2"].isin(["chr10_1102"])
    ],
    chr10q[
        chr10q["CLUSTER_H1"].isin(["chr10_968", "chr10_1088", "chr10_1108", "chr10_1110"]) & 
        chr10q["CLUSTER_H2"].isin(["chr10_968", "chr10_1088", "chr10_1108", "chr10_1110"])
    ]
]).drop_duplicates(subset="SAMPLE_TUMOR")

In [ ]:
### distant
distant = pd.concat([
    chr4p[
        chr4p["CLUSTER_H1"].isin(["chr4_1072", "chr4_1075", "chr4_1079"]) & 
        chr4p["CLUSTER_H2"].isin(["chr4_1072", "chr4_1075", "chr4_1079"])
    ],
    chr4q[
        chr4q["CLUSTER_H1"].isin(["chr4_1081"]) & 
        chr4q["CLUSTER_H2"].isin(["chr4_1081"])
    ],
    chr5p[
        chr5p["CLUSTER_H1"].isin(["chr5_1103"]) & 
        chr5p["CLUSTER_H2"].isin(["chr5_1103"])
    ],
    chr5q[
        chr5q["CLUSTER_H1"].isin(["chr5_1106"]) & 
        chr5q["CLUSTER_H2"].isin(["chr5_1106"])
    ],
    chr10p[
        chr10p["CLUSTER_H1"].isin(["chr10_968", "chr10_1088", "chr10_1108", "chr10_1110"]) & 
        chr10p["CLUSTER_H2"].isin(["chr10_968", "chr10_1088", "chr10_1108", "chr10_1110"])
    ],
    chr10q[
        chr10q["CLUSTER_H1"].isin(["chr10_1102", "chr10_1114"]) & 
        chr10q["CLUSTER_H2"].isin(["chr10_1102", "chr10_1114"])
    ]
]).drop_duplicates(subset="SAMPLE_TUMOR")

In [ ]:
other = pd.concat([chr4p, chr4q, chr5q, chr10p, chr10q]).drop_duplicates(subset="SAMPLE_TUMOR")
col_other = []
for sample in other["SAMPLE_TUMOR"].values:
    if sample not in close["SAMPLE_TUMOR"].values and sample not in distant["SAMPLE_TUMOR"].values:
        col_other.append("PASS")
    else:
        col_other.append("FAIL")
other["STATUS"] = col_other
other = other[other["STATUS"] == "PASS"]

In [ ]:
### make dataframe for plotting
data_for_plot = []

for dataframe, label in zip([close, distant, other], ["close", "distant", "other"]):
    for _, row in dataframe.iterrows():
        data_for_plot.append((label, row["CHROM"], row["SCALED_F"], row["SAMPLE_TUMOR"]))

plot_df = pd.DataFrame(data_for_plot, columns=["CATEGORY", "CHROM", "SCALED_F", "SAMPLE_TUMOR"])

In [ ]:
fig, ax = plt.subplots()
sns.boxplot(
    plot_df,
    x="CATEGORY",
    y="SCALED_F",
)

sns.swarmplot(
    plot_df,
    x="CATEGORY",
    y="SCALED_F",
    size=7
)

ax.set_ylabel("Number of centromeres in the tumor cell")
ax.set_xlabel("CDR position relative to arm loss")

plt.tight_layout()
#plt.savefig("../plotting/plots/pcawg_chr4_5_10_arm_level_detail_analysis.png")

In [ ]:
new_close = [
    "06dad93a-4b67-49b3-8fa5-f2546e22bae7", 
    "1ff155dd-deb4-44e0-b3a2-e4875b618435",
    "f8733b80-c622-11e3-bf01-24c6515278c0",
    "fc962061-8ab0-42b3-e040-11ac0c4810cc",
    "0bfd1043-817c-e3e4-e050-11ac0c4860c5"
]
new_distant = [
    "d0e033bc-8e97-4322-b3b1-8626f1e2449b",
    "ec31e921-0782-4f77-ac23-4190af1e61ee",
    "f4e673a9-5b85-46e7-bcac-2ce6e1026896",
    "ffad9288-c622-11e3-bf01-24c6515278c0",
    "2fe2433b-9a55-431f-a869-2b5a63e8e8fc",
    "7fc7b316-f3d3-4e71-9417-423fe334e6bc"

]
col_reclassify = []
for _, row in plot_df.iterrows():
    if row["SAMPLE_TUMOR"] in new_close:
        col_reclassify.append("close")
    elif row["SAMPLE_TUMOR"] in new_distant:
        col_reclassify.append("distant")
    else:
        col_reclassify.append(row["CATEGORY"])
plot_df["RECLASS_CATEGORY"] = col_reclassify

In [ ]:
stats.kstest(
    plot_df[plot_df["CATEGORY"].eq("close")]["SCALED_F"],
    plot_df[plot_df["CATEGORY"].eq("distant")]["SCALED_F"],
).pvalue

In [ ]:
ax_data = plt.subplot(1, 2, 1)
ax_data_manuell = plt.subplot(1, 2, 2)

plt.gcf().set_size_inches(8, 4)

sns.boxplot(
    plot_df,
    x="CATEGORY",
    y="SCALED_F",
    ax=ax_data,
    color="lightgray",
    linewidth=0,  # removes only the box outline
    whiskerprops={"color": "black", "linewidth": 1.5},
    capprops={"color": "black", "linewidth": 1.5},
    medianprops={"color": "black", "linewidth": 1.5},
)

sns.swarmplot(
    plot_df,
    x="CATEGORY",
    y="SCALED_F",
    size=7,
    ax=ax_data
)

sns.boxplot(
    plot_df,
    x="RECLASS_CATEGORY",
    y="SCALED_F",
    ax=ax_data_manuell,
    color="lightgray",
    linewidth=0,  # removes only the box outline
    whiskerprops={"color": "black", "linewidth": 1.5},
    capprops={"color": "black", "linewidth": 1.5},
    medianprops={"color": "black", "linewidth": 1.5},
)

sns.swarmplot(
    plot_df,
    x="RECLASS_CATEGORY",
    y="SCALED_F",
    size=7,
    hue="CATEGORY",
    ax=ax_data_manuell
)

pv = stats.kstest(
    plot_df[plot_df["CATEGORY"].eq("close")]["SCALED_F"],
    plot_df[plot_df["CATEGORY"].eq("distant")]["SCALED_F"],
).pvalue
print("K-S test p-value for CATEGORY close vs distant:", pv)
pv = stats.kstest(
    plot_df[plot_df["RECLASS_CATEGORY"].eq("close")]["SCALED_F"],
    plot_df[plot_df["RECLASS_CATEGORY"].eq("distant")]["SCALED_F"],
).pvalue
print("K-S test p-value for RECLASS_CATEGORY close vs distant:", pv)

print("Small F, original, close")
print(plot_df[plot_df["RECLASS_CATEGORY"].eq("close") & plot_df["SCALED_F"].between(0, 1.5)].shape[0] / plot_df[plot_df["RECLASS_CATEGORY"].eq("close")].shape[0])
print("Small F, original, distant")
print(plot_df[plot_df["RECLASS_CATEGORY"].eq("distant") & plot_df["SCALED_F"].between(0, 1.5)].shape[0] / plot_df[plot_df["RECLASS_CATEGORY"].eq("distant")].shape[0])
print("Small F, reclassify, close")
print(plot_df[plot_df["CATEGORY"].eq("close") & plot_df["SCALED_F"].between(0, 1.5)].shape[0] / plot_df[plot_df["CATEGORY"].eq("close")].shape[0])
print("Small F, reclassify, distant")
print(plot_df[plot_df["CATEGORY"].eq("distant") & plot_df["SCALED_F"].between(0, 1.5)].shape[0] / plot_df[plot_df["CATEGORY"].eq("distant")].shape[0])

for ax in [ax_data, ax_data_manuell]:
    ax.set_ylabel("# centromeres in the tumor cell")
    ax.set_xlabel("CDR position relative to arm loss")
    ax.set_ylim(0, 3)
    plt.tight_layout()
    
ax_data.set_title("Original configuration")
ax_data_manuell.set_title("Reclassified configuration")

plt.show()

#### Analyse architecture *k*-mer for the chr5 samples with unclear CDR position to find out which architecture was lost

In [ ]:
pcawg_normalization_df = pd.read_csv("/g/korbel/hain/centromere/cancer/PCAWG_normalization.tsv", sep="\t").set_index("SAMPLE")
pcawg_normalization_df.head()

#### chr5

In [ ]:
chr5_other = other[other["CHROM"] == "chr5"]#[["SAMPLE_TUMOR", "SAMPLE_GERMLINE", "SCALED_F", "CLUSTER_H1", "CLUSTER_H2"]]
chr5_other_samples = chr5_other["SAMPLE_TUMOR"].unique().tolist() + chr5_other["SAMPLE_GERMLINE"].unique().tolist()

In [ ]:
### load pcawg chr5 KM data and 
chr5_pcawg_km = pd.read_csv(
    "/g/korbel/hain/centromere/storage/scratch/hain/centromere/aggregated_kmer_tables/chr5_PCAWG.kmer_km.tsv", 
    sep="\t",
    usecols=["CLUSTER"] + [f"{x}_KM" for x in chr5_other_samples],
)

In [ ]:
from scipy.stats import ttest_ind, ks_2samp

In [ ]:
for idx, row in chr5_other.iterrows():
    
    tumor_sample = row["SAMPLE_TUMOR"]
    germline_sample = row["SAMPLE_GERMLINE"]
    f = np.round(row["SCALED_F"], 2)
    purity = row["PURITY"]
    chrom = "chr5"
    cn = row["CN"]
    
    if f >= 1.8:
        continue
    
    print(tumor_sample)
    
    # take normalization set from the CN=2 arm
    if cn == "2_1":
        norm_set = "NORM_P"
    elif cn == "1_2":
        norm_set = "NORM_Q"
    norm_tumor_local = chr5_pcawg_km[chr5_pcawg_km["CLUSTER"].str.contains(norm_set)][f"{tumor_sample}_KM"].mean(skipna=True)
    norm_germline_local = chr5_pcawg_km[chr5_pcawg_km["CLUSTER"].str.contains(norm_set)][f"{germline_sample}_KM"].mean(skipna=True)
    
    norm_tumor_global = pcawg_normalization_df.loc[tumor_sample, "NORMALIZATION"]
    norm_germline_global = pcawg_normalization_df.loc[germline_sample, "NORMALIZATION"]
    
    cluster_h1 = row["CLUSTER_H1"].split("_")[1]
    cluster_h2 = row["CLUSTER_H2"].split("_")[1]
    
    if cluster_h1 in ["1106", "1103"] or cluster_h2 in ["1106", "1103"]:
        pass
    else:
        continue

    kmer_group_h1 = f"{chrom}_{cluster_h1}"
    kmer_group_h2 = f"{chrom}_{cluster_h2}"
    
    if pd.isna(cluster_h1) or pd.isna(cluster_h2):
        continue
    
    fig, ax = plt.subplots()

    ax.set_title(f"{tumor_sample}\nF={f} p={purity} CN={cn}")
    
    
    plotting_data = []

    ### iterate over kmer groups and normalization factors
    for kmers, norm_tumor, norm_germline, label in zip(
        [kmer_group_h1, kmer_group_h2, kmer_group_h1, kmer_group_h2], 
        [norm_tumor_local, norm_tumor_local, norm_tumor_global, norm_tumor_global],
        [norm_germline_local, norm_germline_local, norm_germline_global, norm_germline_global],
        ["LOCAL", "LOCAL", "GLOBAL", "GLOBAL"]
        ):
    
        tumor_km = chr5_pcawg_km[chr5_pcawg_km["CLUSTER"] == kmers][f"{tumor_sample}_KM"].to_numpy(dtype=float) / norm_tumor
        germline_km = chr5_pcawg_km[chr5_pcawg_km["CLUSTER"] == kmers][f"{germline_sample}_KM"].to_numpy(dtype=float) / norm_germline
        
        km_df = pd.DataFrame({
            "tumor_km": tumor_km,
            "germline_km": germline_km
        })
        km_df = km_df[km_df.max(axis=1) > 0]
        km_df["FRACTION"] = km_df["tumor_km"] / km_df["germline_km"]
        km_df["CLUSTER"] = kmers
        km_df["NORM"] = label
    
        plotting_data.append(km_df)
    
    plotting_data_df = pd.concat(plotting_data).reset_index(drop=True)
    sns.boxplot(x="NORM", y="FRACTION", hue="CLUSTER", data=plotting_data_df, showfliers=False)
        
    #ax.plot([0, max_tumor_km], [0, max_tumor_km])
    #ax.legend()
    ax.plot([-.5, 1.5], [1, 1], linestyle="--", color="gray")
    plt.tight_layout()
    plt.show()

#### chr10

In [ ]:
chr10_other = other[other["CHROM"] == "chr10"]#[["SAMPLE_TUMOR", "SAMPLE_GERMLINE", "SCALED_F", "CLUSTER_H1", "CLUSTER_H2"]]
chr10_other_samples = chr10_other["SAMPLE_TUMOR"].unique().tolist() + chr10_other["SAMPLE_GERMLINE"].unique().tolist()

In [ ]:
### load pcawg chr10 KM data and 
chr10_pcawg_km = pd.read_csv(
    "/g/korbel/hain/centromere/storage/scratch/hain/centromere/aggregated_kmer_tables/chr10_PCAWG.kmer_km.tsv", 
    sep="\t",
    usecols=["CLUSTER"] + [f"{x}_KM" for x in chr10_other_samples],
)

In [ ]:
for idx, row in chr10_other.iterrows():
    
    tumor_sample = row["SAMPLE_TUMOR"]
    germline_sample = row["SAMPLE_GERMLINE"]
    f = np.round(row["SCALED_F"], 2)
    purity = row["PURITY"]
    chrom = "chr10"
    cn = row["CN"]
    
    if f >= 1.8:
        continue
    
    
    
    # take normalization set from the CN=2 arm
    if cn == "2_1":
        norm_set = "NORM_P"
    elif cn == "1_2":
        norm_set = "NORM_Q"
    norm_tumor_local = chr10_pcawg_km[chr10_pcawg_km["CLUSTER"].str.contains(norm_set)][f"{tumor_sample}_KM"].mean(skipna=True)
    norm_germline_local = chr10_pcawg_km[chr10_pcawg_km["CLUSTER"].str.contains(norm_set)][f"{germline_sample}_KM"].mean(skipna=True)
    
    norm_tumor_global = pcawg_normalization_df.loc[tumor_sample, "NORMALIZATION"]
    norm_germline_global = pcawg_normalization_df.loc[germline_sample, "NORMALIZATION"]
    
    cluster_h1 = row["CLUSTER_H1"].split("_")[1]
    cluster_h2 = row["CLUSTER_H2"].split("_")[1]
    
    if cluster_h1 in ["1102", "968", "1088", "1108", "1110"] or cluster_h2 in ["1102", "968", "1088", "1108", "1110"]:
        pass
    else:
        continue

    kmer_group_h1 = f"{chrom}_{cluster_h1}"
    kmer_group_h2 = f"{chrom}_{cluster_h2}"
    
    if pd.isna(cluster_h1) or pd.isna(cluster_h2):
        continue
    
    print(tumor_sample)
    
    fig, ax = plt.subplots()

    ax.set_title(f"{tumor_sample}\nF={f} p={purity} CN={cn}")
    
    
    plotting_data = []

    ### iterate over kmer groups and normalization factors
    for kmers, norm_tumor, norm_germline, label in zip(
        [kmer_group_h1, kmer_group_h2, kmer_group_h1, kmer_group_h2], 
        [norm_tumor_local, norm_tumor_local, norm_tumor_global, norm_tumor_global],
        [norm_germline_local, norm_germline_local, norm_germline_global, norm_germline_global],
        ["LOCAL", "LOCAL", "GLOBAL", "GLOBAL"]
        ):
    
        tumor_km = chr10_pcawg_km[chr10_pcawg_km["CLUSTER"] == kmers][f"{tumor_sample}_KM"].to_numpy(dtype=float) / norm_tumor
        germline_km = chr10_pcawg_km[chr10_pcawg_km["CLUSTER"] == kmers][f"{germline_sample}_KM"].to_numpy(dtype=float) / norm_germline
        
        km_df = pd.DataFrame({
            "tumor_km": tumor_km,
            "germline_km": germline_km
        })
        km_df = km_df[km_df.max(axis=1) > 0]
        km_df["FRACTION"] = km_df["tumor_km"] / km_df["germline_km"]
        km_df["CLUSTER"] = kmers
        km_df["NORM"] = label
    
        plotting_data.append(km_df)
    
    plotting_data_df = pd.concat(plotting_data).reset_index(drop=True)
    sns.boxplot(x="NORM", y="FRACTION", hue="CLUSTER", data=plotting_data_df, showfliers=False)
    
    #ax.plot([0, max_tumor_km], [0, max_tumor_km])
    #ax.legend()
    ax.plot([-.5, 1.5], [1, 1], linestyle="--", color="gray")
    plt.tight_layout()
    plt.show()

#### chr4

In [ ]:
chr4_other = other[other["CHROM"] == "chr4"]#[["SAMPLE_TUMOR", "SAMPLE_GERMLINE", "SCALED_F", "CLUSTER_H1", "CLUSTER_H2"]]
chr4_other_samples = chr4_other["SAMPLE_TUMOR"].unique().tolist() + chr4_other["SAMPLE_GERMLINE"].unique().tolist()

In [ ]:
### load pcawg chr10 KM data and 
chr4_pcawg_km = pd.read_csv(
    "/g/korbel/hain/centromere/storage/scratch/hain/centromere/aggregated_kmer_tables/chr4_PCAWG.kmer_km.tsv", 
    sep="\t",
    usecols=["CLUSTER"] + [f"{x}_KM" for x in chr4_other_samples],
)

In [ ]:
for idx, row in chr4_other.iterrows():
    
    tumor_sample = row["SAMPLE_TUMOR"]
    germline_sample = row["SAMPLE_GERMLINE"]
    f = np.round(row["SCALED_F"], 2)
    purity = row["PURITY"]
    chrom = "chr4"
    cn = row["CN"]
    
    if f >= 1.8:
        continue
    
    # take normalization set from the CN=2 arm
    if cn == "2_1":
        norm_set = "NORM_P"
    elif cn == "1_2":
        norm_set = "NORM_Q"
    norm_tumor_local = chr4_pcawg_km[chr4_pcawg_km["CLUSTER"].str.contains(norm_set)][f"{tumor_sample}_KM"].mean(skipna=True)
    norm_germline_local = chr4_pcawg_km[chr4_pcawg_km["CLUSTER"].str.contains(norm_set)][f"{germline_sample}_KM"].mean(skipna=True)
    
    norm_tumor_global = pcawg_normalization_df.loc[tumor_sample, "NORMALIZATION"]
    norm_germline_global = pcawg_normalization_df.loc[germline_sample, "NORMALIZATION"]
    
    cluster_h1 = row["CLUSTER_H1"].split("_")[1]
    cluster_h2 = row["CLUSTER_H2"].split("_")[1]
    
    if cluster_h1 in ["1081", "1075", "1079", "1072"] or cluster_h2 in ["1081", "1075", "1079", "1072"]:
        pass
    else:
        continue

    kmer_group_h1 = f"{chrom}_{cluster_h1}"
    kmer_group_h2 = f"{chrom}_{cluster_h2}"
    
    if pd.isna(cluster_h1) or pd.isna(cluster_h2):
        continue
    
    print(tumor_sample)
    
    fig, ax = plt.subplots()

    ax.set_title(f"{tumor_sample}\nF={f} p={purity} CN={cn}")
    
    
    plotting_data = []

    ### iterate over kmer groups and normalization factors
    for kmers, norm_tumor, norm_germline, label in zip(
        [kmer_group_h1, kmer_group_h2, kmer_group_h1, kmer_group_h2], 
        [norm_tumor_local, norm_tumor_local, norm_tumor_global, norm_tumor_global],
        [norm_germline_local, norm_germline_local, norm_germline_global, norm_germline_global],
        ["LOCAL", "LOCAL", "GLOBAL", "GLOBAL"]
        ):
    
        tumor_km = chr4_pcawg_km[chr4_pcawg_km["CLUSTER"] == kmers][f"{tumor_sample}_KM"].to_numpy(dtype=float) / norm_tumor
        germline_km = chr4_pcawg_km[chr4_pcawg_km["CLUSTER"] == kmers][f"{germline_sample}_KM"].to_numpy(dtype=float) / norm_germline
        
        km_df = pd.DataFrame({
            "tumor_km": tumor_km,
            "germline_km": germline_km
        })
        km_df = km_df[km_df.max(axis=1) > 0]
        km_df["FRACTION"] = km_df["tumor_km"] / km_df["germline_km"]
        km_df["CLUSTER"] = kmers
        km_df["NORM"] = label
    
        plotting_data.append(km_df)
    
    plotting_data_df = pd.concat(plotting_data).reset_index(drop=True)
    sns.boxplot(x="NORM", y="FRACTION", hue="CLUSTER", data=plotting_data_df, showfliers=False)
    
    #ax.plot([0, max_tumor_km], [0, max_tumor_km])
    #ax.legend()
    ax.plot([-.5, 1.5], [1, 1], linestyle="--", color="gray")
    plt.tight_layout()
    plt.show()